## 기본구조

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class State(TypedDict):
    counter: int
    message: str

# 노드 정의
def increment(state: State) -> dict:
    """카운터를 증가시키는 노드"""
    return {
        "counter": state["counter"] + 1,
        "message": f"카운터 증가: {state['counter']} -> {state['counter'] + 1}"
    }

def double(state: State) -> dict:
    """카운터를 두 배로 만드는 노드"""
    return {
        "counter": state["counter"] * 2,
        "message": f"카운터 두 배: {state['counter']} -> {state['counter'] * 2}"
    }

# 그래프 생성
graph = StateGraph(State)

# 노드 추가
graph.add_node("increment", increment)
graph.add_node("double", double)

# 엣지 추가 - 기본 패턴
graph.add_edge(START, "increment")     # 시작 -> increment
graph.add_edge("increment", "double")   # increment -> double
graph.add_edge("double", END)          # double -> 종료


## 예제 1: 숫자 크기 비교하기

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# State: 숫자와 결과 저장
class NumberState(TypedDict):
    number: int
    result: str

# Node 1: 큰 숫자 처리
def handle_big_number(state):
    return {"result": f"{state['number']}는 큰 숫자입니다!"}

# Node 2: 작은 숫자 처리
def handle_small_number(state):
    return {"result": f"{state['number']}는 작은 숫자입니다!"}

# 조건 함수: 어디로 갈지 결정
def check_size(state):
    if state["number"] > 10:
        return "big"     # 큰 숫자면 "big"
    else:
        return "small"   # 작은 숫자면 "small"

# 그래프 구성
graph = StateGraph(NumberState)
graph.add_node("big_handler", handle_big_number)
graph.add_node("small_handler", handle_small_number)

# 조건부 엣지 추가
graph.add_conditional_edges(
    START,
    check_size,
    {
        "big": "big_handler",      # "big"이면 big_handler로
        "small": "small_handler"   # "small"이면 small_handler로
    }
)

# 끝으로 연결
graph.add_edge("big_handler", END)
graph.add_edge("small_handler", END)

# 테스트해보기
app = graph.compile()

# 큰 숫자로 테스트
result1 = app.invoke({"number": 15, "result": ""})
print(result1)

# 작은 숫자로 테스트
result2 = app.invoke({"number": 5, "result": ""})
print(result2)


{'number': 15, 'result': '15는 큰 숫자입니다!'}
{'number': 5, 'result': '5는 작은 숫자입니다!'}


## 예제 2: 신호등 시스템

In [3]:
# State: 신호등 색깔과 행동 저장
class TrafficState(TypedDict):
    color: str
    action: str

# Node: 각 색깔에 따른 행동
def red_light(state):
    return {"action": "정지하세요! 🛑"}

def yellow_light(state):
    return {"action": "주의하세요! ⚠️"}

def green_light(state):
    return {"action": "출발하세요! 🟢"}

# 조건 함수: 색깔에 따라 결정
def check_color(state):
    color = state["color"].lower()
    if color == "red":
        return "red"
    elif color == "yellow":
        return "yellow"
    else:
        return "green"

# 그래프 구성
graph = StateGraph(TrafficState)
graph.add_node("red", red_light)
graph.add_node("yellow", yellow_light)
graph.add_node("green", green_light)

# 조건부 엣지 - 3가지 경로
graph.add_conditional_edges(
    START,
    check_color,
    {
        "red": "red",
        "yellow": "yellow",
        "green": "green"
    }
)

# 모든 노드를 END로 연결
graph.add_edge("red", END)
graph.add_edge("yellow", END)
graph.add_edge("green", END)

# 테스트
app = graph.compile()

# 다양한 색깔로 테스트
colors = ["red", "yellow", "green", "blue"]
for color in colors:
    result = app.invoke({"color": color, "action": ""})
    print(f"{color} → {result['action']}")


red → 정지하세요! 🛑
yellow → 주의하세요! ⚠️
green → 출발하세요! 🟢
blue → 출발하세요! 🟢


## 예제 3: 간단한 성적 처리 시스템

In [4]:
# State: 점수와 등급 저장
class GradeState(TypedDict):
    score: int
    grade: str

# Node: 각 등급별 처리
def excellent(state):
    return {"grade": f"점수 {state['score']}점 - 우수! 🌟"}

def good(state):
    return {"grade": f"점수 {state['score']}점 - 양호! 👍"}

def needs_improvement(state):
    return {"grade": f"점수 {state['score']}점 - 더 노력하세요! 💪"}

# 조건 함수: 점수에 따라 등급 결정
def determine_grade(state):
    score = state["score"]
    if score >= 90:
        return "excellent"
    elif score >= 70:
        return "good"
    else:
        return "improve"

# 그래프 구성
graph = StateGraph(GradeState)
graph.add_node("excellent", excellent)
graph.add_node("good", good)
graph.add_node("improve", needs_improvement)

graph.add_conditional_edges(
    START,
    determine_grade,
    {
        "excellent": "excellent",
        "good": "good",
        "improve": "improve"
    }
)

graph.add_edge("excellent", END)
graph.add_edge("good", END)
graph.add_edge("improve", END)

# 테스트
app = graph.compile()

# 다양한 점수로 테스트
scores = [95, 85, 65]
for score in scores:
    result = app.invoke({"score": score, "grade": ""})
    print(result['grade'])


점수 95점 - 우수! 🌟
점수 85점 - 양호! 👍
점수 65점 - 더 노력하세요! 💪
